In [3]:
import uproot
import awkward as ak
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import mplhep as hep
import pyarrow
import os 

In [2]:
file_b = uproot.open("C:/Users/HP/Documents/Practica Profesional/Analisis BosonHiggs/Data Sets/Run2012B_DoubleMuParked.root")
tree_b = file_b['Events']

file_c = uproot.open("C:/Users/HP/Documents/Practica Profesional/Analisis BosonHiggs/Data Sets/Run2012C_DoubleMuParked.root")
tree_c = file_c['Events']

# Listar ramas (variables físicas)

print(tree_b.keys())
print(tree_c.keys())

# Archivos de simulación Monte Carlo 
# Señal: H → ZZ* → 4μ
file_higgs = uproot.open("C:/Users/HP/Documents/Practica Profesional/Analisis BosonHiggs/Data Sets/SMHiggsToZZTo4L.root")
tree_higgs = file_higgs['Events']

# Fondo: ZZ → 4μ
file_zz = uproot.open("C:/Users/HP/Documents/Practica Profesional/Analisis BosonHiggs/Data Sets/ZZTo4mu.root")
tree_zz = file_zz['Events']

['run', 'luminosityBlock', 'event', 'PV_npvs', 'PV_x', 'PV_y', 'PV_z', 'nMuon', 'Muon_pt', 'Muon_eta', 'Muon_phi', 'Muon_mass', 'Muon_charge', 'Muon_pfRelIso03_all', 'Muon_pfRelIso04_all', 'Muon_dxy', 'Muon_dxyErr', 'Muon_dz', 'Muon_dzErr', 'nElectron', 'Electron_pt', 'Electron_eta', 'Electron_phi', 'Electron_mass', 'Electron_charge', 'Electron_pfRelIso03_all', 'Electron_dxy', 'Electron_dxyErr', 'Electron_dz', 'Electron_dzErr', 'MET_pt', 'MET_phi']
['run', 'luminosityBlock', 'event', 'PV_npvs', 'PV_x', 'PV_y', 'PV_z', 'nMuon', 'Muon_pt', 'Muon_eta', 'Muon_phi', 'Muon_mass', 'Muon_charge', 'Muon_pfRelIso03_all', 'Muon_pfRelIso04_all', 'Muon_dxy', 'Muon_dxyErr', 'Muon_dz', 'Muon_dzErr', 'nElectron', 'Electron_pt', 'Electron_eta', 'Electron_phi', 'Electron_mass', 'Electron_charge', 'Electron_pfRelIso03_all', 'Electron_dxy', 'Electron_dxyErr', 'Electron_dz', 'Electron_dzErr', 'MET_pt', 'MET_phi']


In [3]:
# Extraer variables físicas de los tres archivos a dataframes 

# Extraer variables físicas y de control de los archivos
# Datos reales (Run2012B)
df_b = tree_b.arrays(['run', 'luminosityBlock', 'event', 'Muon_pt', 'Muon_eta', 'Muon_phi', 'Muon_charge', 'Muon_pfRelIso04_all', 'Muon_dxy', 'Muon_dz', 'Muon_dxyErr', 'Muon_dzErr'], library="pd")

In [4]:
# Datos reales (Run2012C)
df_c = tree_c.arrays(['run', 'luminosityBlock', 'event', 'Muon_pt', 'Muon_eta', 'Muon_phi', 'Muon_charge', 'Muon_pfRelIso04_all', 'Muon_dxy', 'Muon_dz', 'Muon_dxyErr', 'Muon_dzErr'], library="pd")

In [3]:
# Simulación Higgs MC
df_higgs_4mu = tree_higgs.arrays(['run', 'luminosityBlock', 'event', 'Muon_pt', 'Muon_eta', 'Muon_phi', 'Muon_charge', 'Muon_pfRelIso04_all', 'Muon_dxy', 'Muon_dz', 'Muon_dxyErr', 'Muon_dzErr'], library="pd")

# Simulación fondo ZZ MC
df_zz_4mu = tree_zz.arrays(['run', 'luminosityBlock', 'event', 'Muon_pt', 'Muon_eta', 'Muon_phi', 'Muon_charge', 'Muon_pfRelIso04_all', 'Muon_dxy', 'Muon_dz', 'Muon_dxyErr', 'Muon_dzErr'], library="pd")

In [5]:
# Dividimos las listas en columnas individuales para 4 muones

def extraer_4muones(df):
    df_4mu = df[df["Muon_pt"].apply(len) >= 4].copy()
    
    # Muón 1
    df_4mu['pt1'] = df_4mu['Muon_pt'].apply(lambda x: x[0])
    df_4mu['eta1'] = df_4mu['Muon_eta'].apply(lambda x: x[0])
    df_4mu['phi1'] = df_4mu['Muon_phi'].apply(lambda x: x[0])
    df_4mu['charge1'] = df_4mu['Muon_charge'].apply(lambda x: x[0])
    df_4mu['iso1'] = df_4mu['Muon_pfRelIso04_all'].apply(lambda x: x[0]) 
    df_4mu['dxy1'] = df_4mu['Muon_dxy'].apply(lambda x: x[0])        
    df_4mu['dz1'] = df_4mu['Muon_dz'].apply(lambda x: x[0])
    df_4mu['dzErr1'] = df_4mu['Muon_dzErr'].apply(lambda x: x[0])
    df_4mu['dxyErr1'] = df_4mu['Muon_dxyErr'].apply(lambda x: x[0])    

    # Muón 2
    df_4mu['pt2'] = df_4mu['Muon_pt'].apply(lambda x: x[1])
    df_4mu['eta2'] = df_4mu['Muon_eta'].apply(lambda x: x[1])
    df_4mu['phi2'] = df_4mu['Muon_phi'].apply(lambda x: x[1])
    df_4mu['charge2'] = df_4mu['Muon_charge'].apply(lambda x: x[1])
    df_4mu['iso2'] = df_4mu['Muon_pfRelIso04_all'].apply(lambda x: x[1])
    df_4mu['dxy2'] = df_4mu['Muon_dxy'].apply(lambda x: x[1])
    df_4mu['dz2'] = df_4mu['Muon_dz'].apply(lambda x: x[1])
    df_4mu['dxyErr2'] = df_4mu['Muon_dxyErr'].apply(lambda x: x[1])
    df_4mu['dzErr2'] = df_4mu['Muon_dzErr'].apply(lambda x: x[1])

    # Muón 3 
    df_4mu['pt3'] = df_4mu['Muon_pt'].apply(lambda x: x[2])
    df_4mu['eta3'] = df_4mu['Muon_eta'].apply(lambda x: x[2])
    df_4mu['phi3'] = df_4mu['Muon_phi'].apply(lambda x: x[2])
    df_4mu['charge3'] = df_4mu['Muon_charge'].apply(lambda x: x[2])
    df_4mu['iso3'] = df_4mu['Muon_pfRelIso04_all'].apply(lambda x: x[2])
    df_4mu['dxy3'] = df_4mu['Muon_dxy'].apply(lambda x: x[2])
    df_4mu['dz3'] = df_4mu['Muon_dz'].apply(lambda x: x[2])
    df_4mu['dxyErr3'] = df_4mu['Muon_dxyErr'].apply(lambda x: x[2])
    df_4mu['dzErr3'] = df_4mu['Muon_dzErr'].apply(lambda x: x[2])

    # Muón 4 
    df_4mu['pt4'] = df_4mu['Muon_pt'].apply(lambda x: x[3])
    df_4mu['eta4'] = df_4mu['Muon_eta'].apply(lambda x: x[3])
    df_4mu['phi4'] = df_4mu['Muon_phi'].apply(lambda x: x[3])
    df_4mu['charge4'] = df_4mu['Muon_charge'].apply(lambda x: x[3])
    df_4mu['iso4'] = df_4mu['Muon_pfRelIso04_all'].apply(lambda x: x[3])
    df_4mu['dxy4'] = df_4mu['Muon_dxy'].apply(lambda x: x[3])
    df_4mu['dz4'] = df_4mu['Muon_dz'].apply(lambda x: x[3])
    df_4mu['dxyErr4'] = df_4mu['Muon_dxyErr'].apply(lambda x: x[3])  
    df_4mu['dzErr4'] = df_4mu['Muon_dzErr'].apply(lambda x: x[3])
    
    return df_4mu

In [6]:
# Filtramos y separamos los 4 muones usando la función que ya tienes arriba
df_higgs_4mu = extraer_4muones(df_higgs_4mu)
df_zz_4mu = extraer_4muones(df_zz_4mu)

# Guardamos los resultados de Monte Carlo en archivos parquet limpios
df_higgs_4mu.to_parquet("df_higgs_4mu.parquet", index=False)
df_zz_4mu.to_parquet("df_zz_4mu.parquet", index=False)

print("¡Archivos parquet de Monte Carlo guardados con éxito!")

¡Archivos parquet de Monte Carlo guardados con éxito!


In [7]:
df_b_4mu = extraer_4muones(df_b)


In [8]:
df_c_4mu = extraer_4muones(df_c)

In [9]:
# Guardar los DataFrames ya procesados en archivos Parquet
df_b_4mu.to_parquet("df_b_4mu.parquet", index=False)


In [10]:
df_c_4mu.to_parquet("df_c_4mu.parquet", index=False)


In [1]:
import uproot
import awkward as ak
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import mplhep as hep
import pyarrow.parquet as pq

In [2]:
import pyarrow.parquet as pq

# Leer el archivo con pyarrow
tabla = pq.read_table("df_b_4mu.parquet")
# Quitar la metadata de pandas que está causando el error
tabla_limpia = tabla.replace_schema_metadata(None)
# Convertir a DataFrame de pandas
df_b_4mu = tabla_limpia.to_pandas()

In [3]:
import pyarrow.parquet as pq

# Leemos y limpiamos el archivo C
tabla_c = pq.read_table("df_c_4mu.parquet")
tabla_c_limpia = tabla_c.replace_schema_metadata(None)
df_c_4mu = tabla_c_limpia.to_pandas()

print("¡Listo! df_c_4mu cargado con éxito:", df_c_4mu.shape)

¡Listo! df_c_4mu cargado con éxito: (5003215, 48)


In [5]:
# Sobrescribimos los archivos en tu disco para que se guarden sin la metadata corrupta
df_b_4mu.to_parquet("df_b_4mu.parquet", index=False)
df_c_4mu.to_parquet("df_c_4mu.parquet", index=False)

In [6]:
# Guardas el archivo C ya limpio en tu disco
df_c_4mu.to_parquet("df_c_4mu.parquet", index=False)
print("¡df_c_4mu guardado definitivamente en el disco!")

¡df_c_4mu guardado definitivamente en el disco!


In [ ]:
# Cargas el archivo B y el archivo C
df_b_4mu = pd.read_parquet("df_b_4mu.parquet")
df_c_4mu = pd.read_parquet("df_c_4mu.parquet")


In [3]:
# Leer los archivos individuales limpiando la metadata
df_b_4mu = pq.read_table("df_b_4mu.parquet").replace_schema_metadata(None).to_pandas()
df_c_4mu = pq.read_table("df_c_4mu.parquet").replace_schema_metadata(None).to_pandas()

# Concatenar Run B y Run C en un solo DataFrame
df_data_real_muones_BC = pd.concat([df_b_4mu, df_c_4mu], ignore_index=True)

# Guardar el archivo unificado con el nombre solicitado
df_data_real_muones_BC.to_parquet("df_data_real_muones_BC.parquet", index=False)

print(f"¡Listo! Datos guardados como 'df_data_real_muones_BC.parquet'. Total de eventos combinados: {len(df_data_real_muones_BC)}")

¡Listo! Datos guardados como 'df_data_real_muones_BC.parquet'. Total de eventos combinados: 8343295


In [2]:
import pyarrow.parquet as pq

# 1. Cargar y limpiar el de Higgs
tabla_higgs = pq.read_table("df_higgs_4mu.parquet")
df_4mu_higgs = tabla_higgs.replace_schema_metadata(None).to_pandas()
# Volver a guardarlo limpio
df_4mu_higgs.to_parquet("df_higgs_4mu.parquet", index=False)

# 2. Cargar y limpiar el de ZZ
tabla_zz = pq.read_table("df_zz_4mu.parquet")
df_4mu_zz = tabla_zz.replace_schema_metadata(None).to_pandas()
# Volver a guardarlo limpio
df_4mu_zz.to_parquet("df_zz_4mu.parquet", index=False)

print("¡Archivos de Monte Carlo leídos, limpiados y guardados de nuevo con éxito!")
print("Número de eventos (Higgs MC):", len(df_4mu_higgs))
print("Número de eventos (ZZ MC):", len(df_4mu_zz))

¡Archivos de Monte Carlo leídos, limpiados y guardados de nuevo con éxito!
Número de eventos (Higgs MC): 30534
Número de eventos (ZZ MC): 425182


In [ ]:
# Ver las columnas de tus DataFrames de muones
print("Columnas en df_4mu:", df_4mu.columns.tolist())
print("Columnas en df_4mu_higgs:", df_4mu_higgs.columns.tolist())
print("Columnas en df_4mu_zz:", df_4mu_zz.columns.tolist())